# Causal Uplift Modeling

The previous notebook answered a specific predictive question:

> **Who is most likely to churn without intervention?**

This notebook asks a different causal question:

> **Whose churn outcome is most likely to change because of an intervention?**

Using the randomized treatment data, we will move from average treatment effects to heterogeneous treatment effects and uplift modeling.

The steps:

- establish a causal modeling dataset and evaluation split
- estimate treatment effects with a simple T-Learner
- compare churn risk with estimated treatment response
- evaluate uplift using ranked treatment-control differences
- measure uplift with Qini and AUUC-style diagnostics
- introduce advanced heterogeneous treatment-effect estimators including DRLearner and CausalForestDML
- compare treatment policies based on random targeting, churn risk, and estimated uplift

The goal is to demonstrate that predictive risk and causal responsiveness are different quantities and can lead to different targeting decisions.

In [1]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openml

from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

from econml.metalearners import TLearner
from econml.dr import DRLearner
from econml.dml import CausalForestDML

In [2]:
# load data from OpenML
dataset = openml.datasets.get_dataset(45580)

X, y, categorical_indicator, attribute_names = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute
)

df = X.copy()
df["y"] = y

In [3]:
# basic data setup
target_col = "y"
treatment_col = "t"

feature_cols = [
    col for col in df.columns
    if col not in [target_col, treatment_col]
]

X = df[feature_cols].copy()
y = df[target_col].copy()
treatment = df[treatment_col].copy()

X = X.drop(columns="FACTOR3")

categorical_cols = X.select_dtypes(
    include=["str", "category"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

## Load the Model
I previously trained a catboost model to predict churn. We can load that for when we need churn-risk scores later in this analysis.

In [4]:
model_path = Path("../artifacts/catboost_churn_model.cbm")

churn_risk_model = CatBoostClassifier()
churn_risk_model.load_model(model_path)

CatBoostClassifier(depth=6, eval_metric='PRAUC', iterations=300, l2_leaf_reg=10, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=0)